# Model Training Notebook
### Sales Forecasting & Gen AI Project

## 1. Librariers Imports

In [1]:
import pandas as pd
import numpy as np

C:\Users\shahi\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\shahi\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
pd.set_option('display.max_columns', None)

## 2. Connect to MySQL & Load Feature Table

In [3]:
from urllib.parse import quote_plus

In [4]:
from sqlalchemy import create_engine

In [5]:
DB_USER = "root"
DB_PASSWORD = quote_plus("your password") 
DB_HOST = "localhost"
DB_PORT = 3306
DB_NAME = "you data base name"

connection_string = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

with engine.connect() as conn:
    df = pd.read_sql("SELECT * FROM feature_table", conn)

print("Shape:", df.shape)
df.head()


Shape: (36, 21)


,year,month,total_revenue,order_count,active_customers,quarter,revenue_lag_1,revenue_lag_3,revenue_lag_12,rolling_mean_3,rolling_mean_6,rolling_std_3,mom_growth_pct,yoy_growth_pct,is_festive_month,avg_discount,avg_order_value,repeat_customer_rate,top_category,top_category_share,target_revenue
0,2023,1,6282761.26,243,153,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.029,25854.98,35.39,home & kitchen,0.297,NaN
1,2023,2,5584972.24,198,124,1,6282761.26,NaN,NaN,NaN,NaN,NaN,-11.11,NaN,0,0.040,28206.93,76.77,home & kitchen,0.511,NaN
2,2023,3,7225276.49,244,150,1,5584972.24,NaN,NaN,6.364337e+06,NaN,8.231892e+05,29.37,NaN,0,0.040,29611.79,86.89,home & kitchen,0.423,NaN
3,2023,4,10616603.50,230,147,2,7225276.49,6282761.26,NaN,7.808951e+06,NaN,2.566093e+06,46.94,NaN,0,0.033,46159.15,94.35,sports,0.476,NaN
4,2023,5,12021876.44,262,166,2,10616603.50,5584972.24,NaN,9.954585e+06,NaN,2.465876e+06,13.24,NaN,0,0.032,45885.02,98.09,grocery,0.417,NaN


## 3. Clean Rows with Missing Lag Values

In [6]:
df = df.sort_values(['year', 'month']).reset_index(drop=True)

before_rows = df.shape[0]
df_model = df.dropna().reset_index(drop=True)
after_rows = df_model.shape[0]

print(f"Rows before: {before_rows} | Rows after dropping NaN: {after_rows}")
df_model.head()


Rows before: 36 | Rows after dropping NaN: 24


,year,month,total_revenue,order_count,active_customers,quarter,revenue_lag_1,revenue_lag_3,revenue_lag_12,rolling_mean_3,rolling_mean_6,rolling_std_3,mom_growth_pct,yoy_growth_pct,is_festive_month,avg_discount,avg_order_value,repeat_customer_rate,top_category,top_category_share,target_revenue
0,2024,1,6528003.42,276,166,1,13741610.83,9224626.01,6282761.26,1.051654e+07,9.960742e+06,3.666906e+06,-52.49,3.90,0,0.070,23652.19,100.0,home & kitchen,0.383,2000000.0
1,2024,2,5266690.87,217,141,1,6528003.42,11280009.98,5584972.24,8.512102e+06,8.974701e+06,4.572587e+06,-19.32,-5.70,0,0.029,24270.46,100.0,home & kitchen,0.294,2125000.0
2,2024,3,8084247.51,253,152,1,5266690.87,13741610.83,7225276.49,6.626314e+06,9.020865e+06,1.411349e+06,53.50,11.89,0,0.047,31953.55,100.0,home & kitchen,0.360,2500000.0
3,2024,4,8383036.20,250,157,2,8084247.51,6528003.42,10616603.50,7.244658e+06,8.880600e+06,1.719472e+06,3.70,-21.04,0,0.034,33532.14,100.0,home & kitchen,0.420,2500000.0
4,2024,5,8797381.85,263,162,2,8383036.20,5266690.87,12021876.44,8.421555e+06,8.466828e+06,3.581242e+05,4.94,-26.82,0,0.094,33450.12,100.0,home & kitchen,0.405,2500000.0


## 4. Define Features (X) and Target (y)

In [7]:
feature_columns = [
    'month', 'quarter',
    'revenue_lag_1', 'revenue_lag_3', 'revenue_lag_12',
    'rolling_mean_3', 'rolling_mean_6', 'rolling_std_3',
    'mom_growth_pct', 'yoy_growth_pct',
    'avg_discount', 'order_count', 'avg_order_value',
    'active_customers', 'repeat_customer_rate',
    'top_category_share', 'is_festive_month'
]

X = df_model[feature_columns]
y = df_model['total_revenue']

print("X shape:", X.shape, "| y shape:", y.shape)
X.head()


X shape: (24, 17) | y shape: (24,)


,month,quarter,revenue_lag_1,revenue_lag_3,revenue_lag_12,rolling_mean_3,rolling_mean_6,rolling_std_3,mom_growth_pct,yoy_growth_pct,avg_discount,order_count,avg_order_value,active_customers,repeat_customer_rate,top_category_share,is_festive_month
0,1,1,13741610.83,9224626.01,6282761.26,1.051654e+07,9.960742e+06,3.666906e+06,-52.49,3.90,0.070,276,23652.19,166,100.0,0.383,0
1,2,1,6528003.42,11280009.98,5584972.24,8.512102e+06,8.974701e+06,4.572587e+06,-19.32,-5.70,0.029,217,24270.46,141,100.0,0.294,0
2,3,1,5266690.87,13741610.83,7225276.49,6.626314e+06,9.020865e+06,1.411349e+06,53.50,11.89,0.047,253,31953.55,152,100.0,0.360,0
3,4,2,8084247.51,6528003.42,10616603.50,7.244658e+06,8.880600e+06,1.719472e+06,3.70,-21.04,0.034,250,33532.14,157,100.0,0.420,0
4,5,2,8383036.20,5266690.87,12021876.44,8.421555e+06,8.466828e+06,3.581242e+05,4.94,-26.82,0.094,263,33450.12,162,100.0,0.405,0


## 5. Train-Test Split

In [8]:
test_size = 6

X_train = X.iloc[:-test_size]
X_test  = X.iloc[-test_size:]
y_train = y.iloc[:-test_size]
y_test  = y.iloc[-test_size:]

print("Train size:", X_train.shape[0], "| Test size:", X_test.shape[0])


Train size: 18 | Test size: 6


## 6. Reusable Accuracy Function

In [9]:
from sklearn.metrics import mean_absolute_percentage_error, root_mean_squared_error

In [10]:
results = {}

def evaluate_model(model_name, y_true, y_pred):
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    rmse = root_mean_squared_error(y_true, y_pred)

    results[model_name] = {'MAPE (%)': round(mape, 2), 'RMSE': round(rmse, 2)}

    print(f"{model_name}")
    print(f"  MAPE: {mape:.2f}%")
    print(f"  RMSE: {rmse:.2f}")

    return mape, rmse


## 7. Baseline Model — Naive Seasonal

In [11]:
baseline_pred = X_test['revenue_lag_12'].values

In [12]:
evaluate_model("Baseline (Naive Seasonal)", y_test.values, baseline_pred)

Baseline (Naive Seasonal)
  MAPE: 17.84%
  RMSE: 2748209.13


(17.835843968720518, 2748209.1280557094)

## 8. Linear Regression


In [13]:
from sklearn.linear_model import LinearRegression

In [14]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

In [15]:
evaluate_model("Linear Regression", y_test.values, lr_pred)

Linear Regression
  MAPE: 2.37%
  RMSE: 344677.89


(2.368584584607732, 344677.886920306)

## 9. XGBoost

In [16]:
from xgboost import XGBRegressor

In [17]:
xgb_simple = XGBRegressor(random_state=42)
xgb_simple.fit(X_train, y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [18]:
# Predictions
xgb_simple_pred = xgb_simple.predict(X_test)

In [19]:
evaluate_model("XGBoost (Simple)", y_test.values, xgb_simple_pred)

XGBoost (Simple)
  MAPE: 23.60%
  RMSE: 3237992.12


(23.598132399165173, 3237992.118964892)

## XGBoost — Hyperparameter Tuning

In [20]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

In [21]:
xgb_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [2, 3, 4],
    'learning_rate': [0.05, 0.1, 0.2]
}

tscv = TimeSeriesSplit(n_splits=3)

In [22]:
xgb_search = GridSearchCV(
    estimator=XGBRegressor(random_state=42),
    param_grid=xgb_param_grid,
    cv=tscv,
    scoring='neg_mean_absolute_percentage_error',
    n_jobs=-1
)

In [23]:
xgb_search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBRegressor(...ree=None, ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.05, 0.1, ...], 'max_depth': [2, 3, ...], 'n_estimators': [50, 100, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_absolute_percentage_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",TimeSeriesSpl...est_size=None)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support f

In [24]:
print("Best XGBoost parameters:", xgb_search.best_params_)

Best XGBoost parameters: {'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 200}


In [25]:
best_xgb_model = xgb_search.best_estimator_
xgb_pred = best_xgb_model.predict(X_test)

In [26]:
evaluate_model("XGBoost (Tuned)", y_test.values, xgb_pred)

XGBoost (Tuned)
  MAPE: 22.63%
  RMSE: 3003295.34


(22.633490757021782, 3003295.3358936813)

## 10. Random Forest — Hyperparameter Tuning

In [27]:
from sklearn.ensemble import RandomForestRegressor

In [28]:
rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7, None],
    'min_samples_split': [2, 4]
}

In [29]:
rf_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=rf_param_grid,
    cv=tscv,
    scoring='neg_mean_absolute_percentage_error',
    n_jobs=-1
)

In [30]:
rf_search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestR...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [3, 5, ...], 'min_samples_split': [2, 4], 'n_estimators': [50, 100, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_absolute_percentage_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",TimeSeriesSpl...est_size=None)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for calla

In [31]:
print("Best Random Forest parameters:", rf_search.best_params_)

Best Random Forest parameters: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 100}


In [32]:
best_rf_model = rf_search.best_estimator_
rf_pred = best_rf_model.predict(X_test)

In [33]:
evaluate_model("Random Forest (Tuned)", y_test.values, rf_pred)

Random Forest (Tuned)
  MAPE: 5.37%
  RMSE: 665990.96


(5.370219251232253, 665990.9615964459)

## 11. Compare All Models

In [34]:
comparison_df = pd.DataFrame(results).T
comparison_df = comparison_df.sort_values('MAPE (%)')

print("Model Comparison (best model on top):")
comparison_df


Model Comparison (best model on top):


,MAPE (%),RMSE
Linear Regression,2.37,344677.89
Random Forest (Tuned),5.37,665990.96
Baseline (Naive Seasonal),17.84,2748209.13
XGBoost (Tuned),22.63,3003295.34
XGBoost (Simple),23.60,3237992.12


## 12. Feature Importance (Best Model)

In [71]:
best_model_name = comparison_df.index[0]
print("Best model:", best_model_name)

Best model: Linear Regression


In [36]:
from sklearn.linear_model import Ridge, Lasso

In [37]:
ridge_grid = {'alpha': [0.01, 0.1, 1, 10, 100]}
ridge_search = GridSearchCV(Ridge(), ridge_grid, cv=tscv, scoring='neg_mean_absolute_percentage_error')

In [39]:
ridge_search.fit(X_train, y_train)

C:\Users\shahi\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:264: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 7.41994968417795e-18.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
C:\Users\shahi\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:264: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 5.417353286048567e-17.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
C:\Users\shahi\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:264: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 3.644905825569748e-17.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Ridge()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'alpha': [0.01, 0.1, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_absolute_percentage_error'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",TimeSeriesSpl...est_size=None)
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Controls the verbosity of infor

In [40]:
print("Best Ridge alpha:", ridge_search.best_params_)

Best Ridge alpha: {'alpha': 100}


In [41]:
evaluate_model("Ridge (Tuned)", y_test.values, ridge_search.best_estimator_.predict(X_test))

Ridge (Tuned)
  MAPE: 2.81%
  RMSE: 398593.65


(2.8137842799983255, 398593.64963326656)

In [42]:
lasso_grid = {'alpha': [0.01, 0.1, 1, 10, 100]}
lasso_search = GridSearchCV(Lasso(max_iter=5000), lasso_grid, cv=tscv, scoring='neg_mean_absolute_percentage_error')

In [43]:
lasso_search.fit(X_train, y_train)

C:\Users\shahi\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.669030e+10, tolerance: 1.513e+10
  model = cd_fast.enet_coordinate_descent(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Lasso(max_iter=5000)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'alpha': [0.01, 0.1, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_absolute_percentage_error'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",TimeSeriesSpl...est_size=None)
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Controls the verbo

In [44]:
print("Best Lasso alpha:", lasso_search.best_params_)

Best Lasso alpha: {'alpha': 100}


In [45]:
evaluate_model("Lasso (Tuned)", y_test.values, lasso_search.best_estimator_.predict(X_test))

Lasso (Tuned)
  MAPE: 4.11%
  RMSE: 546184.26


(4.11091663112356, 546184.2611286729)

In [47]:
comparison_df = pd.DataFrame(results).T
comparison_df = comparison_df.sort_values('MAPE (%)')

In [49]:
print("Model Comparison (best model on top):")
comparison_df

Model Comparison (best model on top):


,MAPE (%),RMSE
Linear Regression,2.37,344677.89
Ridge (Tuned),2.81,398593.65
Lasso (Tuned),4.11,546184.26
Random Forest (Tuned),5.37,665990.96
Baseline (Naive Seasonal),17.84,2748209.13
XGBoost (Tuned),22.63,3003295.34
XGBoost (Simple),23.60,3237992.12


In [50]:
import pickle 

In [53]:
pickle.dump(lr_model,open("sales_forecast_model_LR.pkl",'wb'))

In [54]:
df_model

,year,month,total_revenue,order_count,active_customers,quarter,revenue_lag_1,revenue_lag_3,revenue_lag_12,rolling_mean_3,rolling_mean_6,rolling_std_3,mom_growth_pct,yoy_growth_pct,is_festive_month,avg_discount,avg_order_value,repeat_customer_rate,top_category,top_category_share,target_revenue
0,2024,1,6528003.42,276,166,1,13741610.83,9224626.01,6282761.26,1.051654e+07,9.960742e+06,3.666906e+06,-52.49,3.90,0,0.070,23652.19,100.0,home & kitchen,0.383,2000000.0
1,2024,2,5266690.87,217,141,1,6528003.42,11280009.98,5584972.24,8.512102e+06,8.974701e+06,4.572587e+06,-19.32,-5.70,0,0.029,24270.46,100.0,home & kitchen,0.294,2125000.0
2,2024,3,8084247.51,253,152,1,5266690.87,13741610.83,7225276.49,6.626314e+06,9.020865e+06,1.411349e+06,53.50,11.89,0,0.047,31953.55,100.0,home & kitchen,0.360,2500000.0
3,2024,4,8383036.20,250,157,2,8084247.51,6528003.42,10616603.50,7.244658e+06,8.880600e+06,1.719472e+06,3.70,-21.04,0,0.034,33532.14,100.0,home & kitchen,0.420,2500000.0
4,2024,5,8797381.85,263,162,2,8383036.20,5266690.87,12021876.44,8.421555e+06,8.466828e+06,3.581242e+05,4.94,-26.82,0,0.094,33450.12,100.0,home & kitchen,0.405,2500000.0
5,2024,6,8429387.84,269,160,2,8797381.85,8084247.51,7846609.33,8.536602e+06,7.581458e+06,2.270280e+05,-4.18,7.43,0,0.033,31336.01,100.0,home & kitchen,0.383,2500000.0
6,2024,7,7130056.75,249,160,3,8429387.84,8383036.20,9589366.48,8.118942e+06,7.681800e+06,8.759427e+05,-15.41,-25.65,0,0.077,28634.77,100.0,home & kitchen,0.352,2500000.0
7,2024,8,9381893.40,276,167,3,7130056.75,8797381.85,11182936.14,8.313779e+06,8.367667e+06,1.130361e+06,31.58,-16.11,0,0.034,33992.37,100.0,home & kitchen,0.352,2500000.0
8,2024,9,7895263.74,262,164,3,9381893.40,8429387.84,7807263.08,8.135738e+06,8.336170e+06,1.145017e+06,-15.85,1.13,0,0.088,30134.59,100.0,home & kitchen,0.332,2500000.0
9,2024,10,11531126.90,256,165,4,7895263.74,7130056.75,9224626.01,9.602761e+06,8.860852e+06,1.827967e+06,46.05,25.00,1,0.105,45043.46,100.0,home & kitchen,0.387,3500000.0


# Future Predictions

In [60]:
def predict_future_revenue(model, df, feature_cols, n_months=6):
    """
    Generates recursive future predictions for specified n_months.
    """
    future_preds = []
    current_row = df.iloc[-1:].copy()
    history_revenue = list(df['total_revenue'].values)
    
    for _ in range(n_months):
        # 1. Prediction
        next_pred = model.predict(current_row[feature_cols])[0]
        future_preds.append(next_pred)
        history_revenue.append(next_pred)
        
        # 2. Update Lag & Rolling Features
        current_row['revenue_lag_1'] = history_revenue[-1]
        current_row['revenue_lag_3'] = history_revenue[-3]
        current_row['revenue_lag_12'] = history_revenue[-12]
        
        current_row['rolling_mean_3'] = np.mean(history_revenue[-3:])
        current_row['rolling_mean_6'] = np.mean(history_revenue[-6:])
        current_row['rolling_std_3'] = np.std(history_revenue[-3:], ddof=1)
        
        current_row['mom_growth_pct'] = ((history_revenue[-1] - history_revenue[-2]) / history_revenue[-2]) * 100
        current_row['yoy_growth_pct'] = ((history_revenue[-1] - history_revenue[-13]) / history_revenue[-13]) * 100
        
        # 3. Update Calendar Features
        new_month = (current_row['month'].values[0] % 12) + 1
        current_row['month'] = new_month
        current_row['quarter'] = (new_month - 1) // 3 + 1
        current_row['is_festive_month'] = 1 if new_month in [10, 11, 12] else 0

    return future_preds

In [63]:
future_6_months = predict_future_revenue(lr_model, df_model, feature_columns, n_months=12)

# Print Output in Crores / Lakhs
for i, val in enumerate(future_6_months, 1):
    print(f"Month {i}: ₹{val:,.2f} ({val/1e7:.2f} Cr)")

Month 1: ₹13,648,553.92 (1.36 Cr)
Month 2: ₹13,747,700.25 (1.37 Cr)
Month 3: ₹13,769,167.52 (1.38 Cr)
Month 4: ₹14,036,279.85 (1.40 Cr)
Month 5: ₹12,717,068.84 (1.27 Cr)
Month 6: ₹12,926,435.61 (1.29 Cr)
Month 7: ₹13,679,693.06 (1.37 Cr)
Month 8: ₹13,197,367.91 (1.32 Cr)
Month 9: ₹12,969,030.14 (1.30 Cr)
Month 10: ₹13,274,847.93 (1.33 Cr)
Month 11: ₹13,281,696.61 (1.33 Cr)
Month 12: ₹13,386,132.52 (1.34 Cr)


In [66]:
comparison_export = pd.DataFrame({
    'year': df_model.iloc[-test_size:]['year'].values,
    'month': df_model.iloc[-test_size:]['month'].values,
    'actual_revenue': y_test.values,
    'predicted_revenue': lr_model.predict(X_test)
})

comparison_export.to_csv("actual_vs_predicted.csv", index=False)
print(comparison_export)

   year  month  actual_revenue  predicted_revenue
0  2025      7     12490297.99       1.240227e+07
1  2025      8     12102756.18       1.232699e+07
2  2025      9     10132338.42       9.687579e+06
3  2025     10     11242309.04       1.124428e+07
4  2025     11     12207746.02       1.265612e+07
5  2025     12     14154455.35       1.364855e+07


In [70]:
comparison_df.to_csv("model_comparison.csv")

In [69]:
comparison_df

,MAPE (%),RMSE
Linear Regression,2.37,344677.89
Ridge (Tuned),2.81,398593.65
Lasso (Tuned),4.11,546184.26
Random Forest (Tuned),5.37,665990.96
Baseline (Naive Seasonal),17.84,2748209.13
XGBoost (Tuned),22.63,3003295.34
XGBoost (Simple),23.60,3237992.12
